# 02. R4T 실습: 보상 궤적을 합성 지도 데이터로 컴파일하기

> 이 노트북은 R4T의 1~2단계(보상 기반 fan-out 탐색 → 합성 지도 데이터)를 NumPy로 축소한 **toy reproduction**입니다. 실제 FOLM, Soft-GRPO, LLM 생성, frozen retriever 학습을 포함하지 않으며 논문 결과 재현이 아닙니다. 네트워크·API·GPU를 사용하지 않습니다.

## 학습 목표

1. 한 broad query에서 여러 fan-out trajectory를 만들고 집합 보상으로 순위를 정합니다.
2. 그룹 평균과 표준편차로 GRPO식 상대 advantage를 계산합니다.
3. 고보상 결과를 `(z_q, Z_target)` 합성 학습 쌍으로 변환합니다.
4. 행 순열과 무관한 목표 텐서 및 set loss가 왜 필요한지 실험합니다.

In [1]:
import itertools
import numpy as np

SEED = 260306397
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

def normalize(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

def nearest_ids(queries, contents):
    return np.argmax(normalize(queries) @ normalize(contents).T, axis=1)

def vendi_score(embeddings, eps=1e-12):
    x = normalize(embeddings)
    eigenvalues = np.clip(np.linalg.eigvalsh(x @ x.T), 0.0, None)
    p = eigenvalues / eigenvalues.sum()
    return float(np.exp(-np.sum(p * np.log(p + eps))))

print(f'deterministic seed={SEED}')

deterministic seed=260306397


## 1단계: fan-out trajectory와 상대 advantage

논문은 각 질의에서 `G`개 출력을 샘플링하고 다음처럼 그룹 통계로 advantage를 정규화합니다.

`A_i = (r_i - mean(r_1...r_G)) / (std(r_1...r_G) + epsilon)`

실제 R4T는 이 advantage와 Soft-PPO의 양방향 KL 규제를 사용해 언어 모델 정책을 갱신합니다. 여기서는 정책 학습 대신 다양성이 다른 합성 trajectory를 만들고 고보상 trajectory를 선택하는 과정만 재현합니다.

In [2]:
item_names = np.array([
    'boho_dress', 'festival_boots', 'woven_bag', 'minimal_jacket',
    'formal_suit', 'sport_shoe', 'vintage_hat', 'beach_sandal'
])
database = normalize(np.array([
    [0.90,  0.30,  0.15, 0.05], [0.80,  0.10,  0.50, 0.05],
    [0.75,  0.45,  0.10, 0.05], [0.70, -0.15, -0.20, 0.10],
    [0.50, -0.40, -0.25, 0.20], [0.35, -0.10,  0.75, 0.10],
    [0.65,  0.45, -0.10, 0.20], [0.55,  0.20,  0.55, 0.10],
]))
broad_queries = normalize(np.array([
    [1.00,  0.15,  0.10, 0.05],  # bohemian / exploratory
    [0.45, -0.05,  0.85, 0.05],  # sporty / outdoor
    [0.60, -0.55, -0.20, 0.20],  # formal / minimal
]))

def set_reward(query, fanout):
    ids = nearest_ids(fanout, database)
    reps = database[ids]
    ground = 1.0 - np.linalg.norm(normalize(fanout) - reps, axis=1).mean()
    diversity = vendi_score(reps)
    alignment = np.mean(normalize(fanout) @ normalize(query))
    return float(0.6 * ground + 0.2 * diversity + 0.2 * alignment)

def sample_trajectory_group(query, group_size=8, fanout_size=4):
    # 낮은 group index는 최상위 항목을 반복하고, 높은 index일수록 후보를 더 넓게 씁니다.
    ranking = np.argsort(-(database @ query))
    trajectories = []
    for group_index in range(group_size):
        unique_count = 1 + round(group_index * (fanout_size - 1) / (group_size - 1))
        chosen = np.resize(ranking[:unique_count], fanout_size)
        local_noise = rng.normal(0.0, 0.025, size=(fanout_size, database.shape[1]))
        trajectories.append(normalize(database[chosen] + local_noise))
    return np.stack(trajectories)

trajectory_groups = [sample_trajectory_group(q) for q in broad_queries]
reward_groups = [np.array([set_reward(q, t) for t in group])
                 for q, group in zip(broad_queries, trajectory_groups)]

for query_id, rewards in enumerate(reward_groups):
    best = int(np.argmax(rewards))
    print(f'query {query_id}: rewards={np.round(rewards, 3)} / best={best}')

query 0: rewards=[0.973 0.976 0.98  0.972 1.016 1.01  1.091 1.091] / best=6
query 1: rewards=[0.983 0.977 1.022 1.017 1.044 1.041 1.055 1.048] / best=6
query 2: rewards=[0.973 0.977 1.009 1.017 1.075 1.066 1.094 1.105] / best=7


In [3]:
# 논문 식 (6)의 그룹 상대 advantage를 첫 질의에 적용합니다.
rewards = reward_groups[0]
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
print('advantages:', np.round(advantages, 3))
print('mean/std  :', round(float(advantages.mean()), 8), round(float(advantages.std()), 6))

assert len(advantages) == 8
assert abs(advantages.mean()) < 1e-7
assert np.isclose(advantages.std(), 1.0, atol=1e-6)
assert int(np.argmax(advantages)) == int(np.argmax(rewards))

advantages: [-0.848 -0.798 -0.712 -0.885  0.058 -0.086  1.636  1.633]
mean/std  : 0.0 1.0


## 2단계: 고보상 행동을 `(z_q, Z_target)`으로 저장

R4T의 OAR 설정은 fan-out으로 검색된 콘텐츠 임베딩을 목표 행으로 사용합니다. WSCR 설정은 최적화된 하위 질의 임베딩을 목표 행으로 사용합니다. 아래는 OAR 방식의 작은 예입니다. 각 질의에 대해 가장 높은 보상의 trajectory를 고르고, 그 trajectory가 검색한 콘텐츠를 `Z_target in R^(L x d)`에 담습니다. 실제 논문은 최상위 하나만 고르는 이 toy보다 풍부한 reward-shaped 분포에서 샘플을 수집합니다.

In [4]:
synthetic_pairs = []
for query, group, rewards in zip(broad_queries, trajectory_groups, reward_groups):
    selected_fanout = group[int(np.argmax(rewards))]
    content_ids = nearest_ids(selected_fanout, database)
    z_target = database[content_ids].copy()
    synthetic_pairs.append((query.copy(), z_target))

z_queries = np.stack([pair[0] for pair in synthetic_pairs])
z_targets = np.stack([pair[1] for pair in synthetic_pairs])
print('z_queries:', z_queries.shape)
print('z_targets:', z_targets.shape, '(batch, L, d)')
print('first target items:', item_names[nearest_ids(z_targets[0], database)].tolist())

assert z_queries.shape == (3, 4)
assert z_targets.shape == (3, 4, 4)
assert np.allclose(np.linalg.norm(z_targets, axis=-1), 1.0)

z_queries: (3, 4)
z_targets: (3, 4, 4) (batch, L, d)
first target items: ['boho_dress', 'woven_bag', 'festival_boots', 'minimal_jacket']


## 행 순서는 의미가 없다

집합 `{a,b,c,d}`와 `{d,b,a,c}`는 같은 검색 목표입니다. 논문은 학습 시 `Z_target`의 행을 무작위로 섞어 순열 강건성을 유도합니다. 여기서는 단순한 행별 MSE가 같은 집합을 다르다고 벌점 주는 반면, 모든 작은 순열 중 최적 대응을 찾는 set loss는 0이 되는지 확인합니다. 이 완전 탐색은 `L=4`라서만 가능하며 실제 대규모 학습 구현이 아닙니다.

In [5]:
def exact_permutation_loss(prediction, target):
    # 교육용 O(L!) 완전 탐색입니다. L이 커지면 Hungarian/Sinkhorn류의 대응 방법을 고려합니다.
    losses = []
    for permutation in itertools.permutations(range(len(target))):
        reordered = target[list(permutation)]
        losses.append(np.mean((prediction - reordered) ** 2))
    return float(min(losses))

target = z_targets[0]
permutation = np.array([2, 0, 3, 1])
permuted_target = target[permutation]
ordered_mse = float(np.mean((target - permuted_target) ** 2))
set_mse = exact_permutation_loss(target, permuted_target)

print('row-wise MSE        :', round(ordered_mse, 6))
print('permutation set MSE :', round(set_mse, 12))
print('permutation         :', permutation.tolist())

assert ordered_mse > 0.0
assert set_mse < 1e-12
assert set(item_names[nearest_ids(target, database)]) == set(item_names[nearest_ids(permuted_target, database)])
print('Assertions passed: 순서가 바뀌어도 검색 목표 집합은 같습니다.')

row-wise MSE        : 0.102232
permutation set MSE : 0.0
permutation         : [2, 0, 3, 1]
Assertions passed: 순서가 바뀌어도 검색 목표 집합은 같습니다.


## 예상 출력과 해석

- 각 질의에 대해 8개 보상과 선택된 `best` index가 출력됩니다.
- advantage의 평균은 0, 표준편차는 1입니다.
- 합성 텐서 크기는 `(3, 4, 4)`입니다: 질의 3개, fan-out 방향 4개, 임베딩 차원 4개.
- 고정 순열의 row-wise MSE는 양수지만 permutation set MSE는 0이고 마지막 줄은 `Assertions passed`입니다.

## 한계

- trajectory의 다양성을 손으로 설계했으며 Soft-GRPO로 정책을 학습하지 않았습니다.
- 실제 R4T는 질의마다 128개 샘플(temperature 0.9)을 생성하지만 이 예제는 8개만 사용합니다.
- 가장 높은 보상 하나만 고르면 reward-shaped 분포의 다봉성(mode)을 잃을 수 있습니다.
- 가중합 보상은 reward hacking과 편향을 만들 수 있습니다. 실제 적용에는 분포 검사와 인간 평가가 필요합니다.
- 다음 노트북에서는 이 텐서를 조건부 denoising 대상으로 보고, 생성한 `L`개 방향을 한 번의 배치 최근접 이웃 검색으로 콘텐츠에 매핑합니다.